In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgbm



In [19]:
# job, marital, contact, poutcome 은 순서가 없으니 전부 더미(0/1) 변수로 분리
df_dummies = pd.get_dummies(
    data[['job','marital','contact','poutcome']],
    drop_first=False,   
    prefix_sep='_'
)

edu_map = {'unknown':0, 'primary':1, 'secondary':2, 'tertiary':3}
data['education_ord'] = data['education'].map(edu_map)

df_num = data[['age','balance','campaign','pdays','previous','duration']]
df_bin = data[['housing','loan']].replace({'no':0,'yes':1})

y_col = data[['label']]

df_corr = pd.concat([df_num, df_bin, data[['education_ord']], df_dummies, y_col], axis=1)
corr = df_corr.corr()

# 2) 사이즈와 폰트 스케일 조정
plt.figure(figsize=(20, 16))        
sns.set(font_scale=0.8)             

# 3) heatmap 그리기
ax = sns.heatmap(
    corr,
    cmap='coolwarm',                
    annot=True,                     
    fmt='.2f',                      
    linewidths=0.5,                 
    cbar_kws={"shrink": .5}         
)

# 4) 축 레이블 회전 및 정렬
ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=90,                    
    ha='center'                    
)
ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0,                     
    va='center'                     
)

plt.tight_layout()                  
plt.show()


NameError: name 'data' is not defined

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin


In [3]:
class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, attribute_names):
        self.attribute_names = attribute_names
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.attribute_names]

In [4]:
# 사용자 정의 CategoricalEncoder
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, encoding='onehot-dense', categories='auto', dtype=np.float64, handle_unknown='ignore'):
        self.encoding = encoding
        self.categories = categories
        self.dtype = dtype
        self.handle_unknown = handle_unknown

    def fit(self, X, y=None):
        from sklearn.utils import check_array
        from sklearn.preprocessing import LabelEncoder
        X = check_array(X, dtype=object, accept_sparse='csc', copy=True)
        n_samples, n_features = X.shape
        self._label_encoders_ = [LabelEncoder() for _ in range(n_features)]
        for i in range(n_features):
            Xi = X[:, i]
            le = self._label_encoders_[i]
            if self.categories == 'auto':
                le.fit(Xi)
            else:
                le.classes_ = np.array(sorted(self.categories[i]))
        self.categories_ = [le.classes_ for le in self._label_encoders_]
        return self

    def transform(self, X):
        from sklearn.utils import check_array
        X = check_array(X, dtype=object, accept_sparse='csc', copy=True)
        n_samples, n_features = X.shape
        X_int = np.zeros_like(X, dtype=int)
        mask = np.ones_like(X, dtype=bool)
        for i in range(n_features):
            valid = np.in1d(X[:, i], self.categories_[i])
            if not valid.all():
                if self.handle_unknown == 'error':
                    raise ValueError(f"Unknown categories in column {i}")
                X[~valid, i] = self.categories_[i][0]
            X_int[:, i] = self._label_encoders_[i].transform(X[:, i])
            mask[:, i] = valid
        if self.encoding == 'ordinal':
            return X_int.astype(self.dtype)
        # one-hot
        n_values = [cats.size for cats in self.categories_]
        indices = np.cumsum([0] + n_values)
        row_idx = np.repeat(np.arange(n_samples), n_features)[mask.ravel()]
        col_idx = (X_int + indices[:-1]).ravel()[mask.ravel()]
        data = np.ones_like(col_idx, dtype=self.dtype)
        mat = sparse.csr_matrix((data, (row_idx, col_idx)), shape=(n_samples, indices[-1]))
        if self.encoding == 'onehot-dense':
            return mat.toarray()
        return mat

# DataFrameSelector 정의
class DataFrameSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns): self.columns = columns
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.columns].values

In [5]:
num_cols = ["age", "balance", "day", "campaign", "pdays", "previous","duration"]
cat_cols = ["job", "education", "marital", "default", "housing", "loan", "contact", "month", "poutcome"]

# pipeline
def build_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', CategoricalEncoder(encoding='onehot-dense',
                                      handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

preprocessor = build_preprocessor(num_cols, cat_cols)

### 검증데이터 분리

In [6]:
# 데이터 로드 
train_df = pd.read_csv('train.csv')
X = train_df[num_cols + cat_cols]
y = train_df['label']

#전처리 & 학습/검증 분리
X_trans = preprocessor.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(
    X_trans, y, test_size=0.2, random_state=42, stratify=y
)

/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xj

#### 3.2 초기 모델 정의 및 학습

##### LGBM

In [7]:
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
#학습
base_model.fit(X_train, y_train)

LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

In [8]:
# Feature Names 생성
num_features = num_cols.copy()
cat_pipe = preprocessor.named_transformers_['cat']
encoder  = cat_pipe.named_steps['encode']
categories = encoder.categories_

cat_features = []
for col, cats in zip(cat_cols, categories):
    for cat in cats:
        cat_features.append(f"{col}_{cat}")

feature_names = num_features + cat_features

#### 3.3 변수 중요도 분석

In [28]:
# 10) Feature Importances 추출 & 정리
importances = base_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print(importance_df.to_string(index=False))

            Feature  Importance
           duration         495
            balance         357
                day         330
                age         275
              pdays         166
           campaign         134
          month_nov          66
         housing_no          64
          month_may          62
    contact_unknown          60
           previous          58
          month_oct          55
          month_jul          53
          month_mar          53
          month_feb          53
          month_aug          50
          month_jun          50
          month_apr          45
   poutcome_success          41
            loan_no          40
          month_jan          36
   contact_cellular          35
  education_primary          34
 education_tertiary          34
          month_sep          31
         job_admin.          29
    marital_married          25
education_secondary          23
   poutcome_failure          22
          month_dec          21
     mar

#### 3.4 변수 선택법

In [31]:
from sklearn.feature_selection import SequentialFeatureSelector

# 변수 선택 모델 정의
sfs = SequentialFeatureSelector(
    base_model,
    n_features_to_select='auto',
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)
# 변수 선택 모델 학습하기
sfs.fit(X_train, y_train)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

SequentialFeatureSelector(estimator=LGBMClassifier(class_weight='balanced',
                                                   random_state=42,
                                                   verbose=-1),
                          n_jobs=-1, scoring='f1')

In [32]:
mask = sfs.get_support()  
selected_features = [name for name, keep in zip(feature_names, mask) if keep]
print("선택된 변수:", selected_features)

선택된 변수: ['age', 'balance', 'day', 'pdays', 'duration', 'job_services', 'education_secondary', 'education_tertiary', 'marital_single', 'default_no', 'default_yes', 'housing_no', 'housing_yes', 'contact_cellular', 'month_apr', 'month_aug', 'month_feb', 'month_jan', 'month_jul', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'poutcome_success', 'poutcome_unknown']


In [9]:
#selected_features = ['age', 'balance', 'day', 'pdays', 'duration', 'job_services', 'education_secondary', 'education_tertiary', 'marital_single', 'default_no', 'default_yes', 'housing_no', 'housing_yes', 'contact_cellular', 'month_apr', 'month_aug', 'month_feb', 'month_jan', 'month_jul', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'poutcome_success', 'poutcome_unknown']


In [10]:
# NumPy array → DataFrame 변환
df_train = pd.DataFrame(X_train, columns=feature_names)
df_val   = pd.DataFrame(X_val,   columns=feature_names)
df_test  = pd.DataFrame(preprocessor.transform(pd.read_csv('test.csv')[num_cols+cat_cols]),
                        columns=feature_names)

# 컬럼 필터링
X_train_sel = df_train[selected_features]
X_val_sel   = df_val[selected_features]
X_test_sel  = df_test[selected_features]

# 다시 모델 학습 & 평가
base_model.fit(X_train_sel, y_train)
val_pred = base_model.predict(X_val_sel)


/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xjvj0x4vj84g36qs961pg80000gn/T/ipykernel_50851/3131163131.py:32: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  valid = np.in1d(X[:, i], self.categories_[i])
/var/folders/_g/y6xj

### 하이퍼파라미터 튜닝 (GBM - Random)

In [11]:
param_dist = {
    'n_estimators':       [100, 200, 300, 400, 500],
    'learning_rate':      [0.01, 0.05, 0.1, 0.15, 0.2],
    'class_weight': ['balanced', {0:1, 1:2}, {0:1, 1:5}, {0:1, 1:10}],
    'max_depth':          [3, 4, 5, 6],
    'min_samples_split':  [2, 5, 10, 20],
    'min_samples_leaf':   [1, 2, 5, 10],
    'subsample':          [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features':       ['sqrt', 'log2', None]
}



In [14]:
# 3) GradientBoostingClassifier와 RandomizedSearchCV 설정
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score

random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=50,                    
    scoring=make_scorer(f1_score, pos_level = 1),
    cv=5,                         
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [15]:
random_search.fit(X_train_sel , y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 388, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                  

RandomizedSearchCV(cv=5,
                   estimator=LGBMClassifier(class_weight='balanced',
                                            random_state=42, verbose=-1),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'class_weight': ['balanced',
                                                         {0: 1, 1: 2},
                                                         {0: 1, 1: 5},
                                                         {0: 1, 1: 10}],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.15,
                                                          0.2],
                                        'max_depth': [3, 4, 5, 6],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_leaf': [1, 2, 5, 10],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500],
                                        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]},
                   random_state=42,
                   scoring=make_scorer(f1_score, response_method='predict', pos_level=1),
                   verbose=1)

In [19]:
best_params = random_search.best_params_
best_lgbm = random_search.best_estimator_

In [21]:
print(best_params)

{'subsample': 0.6, 'n_estimators': 500, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 4, 'learning_rate': 0.15, 'class_weight': 'balanced'}


In [22]:
from sklearn.metrics import precision_recall_curve
# 모델 평가 지표
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# 데이터 시각화
import seaborn as sns

In [24]:
# 예측 확률값
X_train_sel = df_train[selected_features]
X_val_sel   = df_val[selected_features]
X_test_sel  = df_test[selected_features]
y_prob = best_lgbm.predict_proba(X_val_sel )[:, 1]  # 양성 클래스 확률

# Precision, Recall, Threshold 계산
precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)

# F1 Score 계산
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

# 최적 Threshold 선택
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best Threshold for F1 Score: {best_threshold}")

# 최적 Threshold 기반 예측
y_pred = (y_prob >= best_threshold).astype(int)

# F1 Score 확인
print(f"Optimized F1 Score: {f1_score(y_val, y_pred)}")


Best Threshold for F1 Score: 0.5761952494496121
Optimized F1 Score: 0.5913229780396357


### 5. 모델 평가

In [25]:
# threshold 설정
thres=0.2677

In [31]:
# 훈련 데이터에 대한 성능 평가
y_retrn_pred_proba = best_lgbm.predict_proba(X_train_sel)[:, 1]
y_retrn_pred_label = (y_retrn_pred_proba > thres).astype(int)

print(f"훈련 데이터 Accuracy: {accuracy_score(y_train, y_retrn_pred_label )}")
print(f"훈련 데이터 Precision: {precision_score(y_train, y_retrn_pred_label)}")
print(f"훈련 데이터 Recall: {recall_score(y_train, y_retrn_pred_label)}")
print(f"훈련 데이터 F1 Score: {f1_score(y_train, y_retrn_pred_label)}")

훈련 데이터 Accuracy: 0.8491132440652526
훈련 데이터 Precision: 0.436613475177305
훈련 데이터 Recall: 0.9976367319378798
훈련 데이터 F1 Score: 0.6073997944501541


검증 데이터로 성능 평가

In [32]:
# 검증 데이터에 대한 성능 평가
y_val_pred_proba = best_lgbm.predict_proba(X_val_sel)[:, 1]
y_val_pred_label = (y_val_pred_proba > thres).astype(int)

print(f"검증 데이터 Accuracy: {accuracy_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 Precision: {precision_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 Recall: {recall_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 F1 Score: {f1_score(y_val, y_val_pred_label)}")

검증 데이터 Accuracy: 0.825434439178515
검증 데이터 Precision: 0.39156268568033276
검증 데이터 Recall: 0.8905405405405405
검증 데이터 F1 Score: 0.543953776310359


In [ ]:
import shap
import matplotlib.pyplot as plt

# 1) TreeExplainer 생성
#    best_model: 이미 학습된 LGBMClassifier 혹은 GradientBoostingClassifier
explainer = shap.TreeExplainer(base_model)

# 2) SHAP 값 계산
#    X_train: 전처리된 훈련용 특성 행렬 (NumPy array 또는 DataFrame)
shap_values = explainer.shap_values(X_train)

# 3) 글로벌 중요도 (bar plot)
shap.summary_plot(
    shap_values, 
    X_train, 
    feature_names=feature_names, 
    plot_type="bar"
)

# 4) 변수별 분포 (beeswarm plot)
shap.summary_plot(
    shap_values, 
    X_train, 
    feature_names=feature_names
)
